### Lecture 2: LAB 1 - Bộ lọc hữu hạn FIR - Phép chập, tương quan và ma trận.


### Phép chập 1 chiều với kernel $ w $:

Công thức tổng quát của phép chập 1 chiều sau:

\begin{equation}
(f * w)[n] = \sum_{m=0}^{M-1} f[m] \cdot w[M - 1 - (n - m)]
\end{equation}

Trong đó:
- $ f $ là tín hiệu đầu vào.
- $ w $ là kernel (filter) được gập ngược trước khi trượt qua tín hiệu.
- $ M $ là độ dài của kernel.

Dưới đây là ví dụ về phép chập 1 chiều với tín hiệu $f = [0, 0, 0, 1, 0, 0, 0, 0] $ và kernel \( w = [1, 2, 3, 2, 8] \).

## CASE: No PADDING
### Bước 1: Gập kernel $ w $
Ta gập kernel $ w $, tức là đảo ngược thứ tự của kernel:

\begin{equation}
w_{\text{gập}} = [8, 2, 3, 2, 1]
\end{equation}

### Bước 2: Thực hiện phép chập

Bây giờ, ta trượt kernel $ w_{\text{gập}} $ qua tín hiệu $ f $ và tính kết quả tại từng vị trí.

#### Tại vị trí $ n = 0 $:
Kernel chỉ trùng với các phần tử không trong tín hiệu:

$$
f[0] \cdot w_{\text{gập}}[0] + f[1] \cdot w_{\text{gập}}[1] + f[2] \cdot w_{\text{gập}}[2] + f[3] \cdot w_{\text{gập}}[3] + f[4] \cdot w_{\text{gập}}[4] = 0 \cdot 8 + 0 \cdot 2 + 0 \cdot 3 + 1 \cdot 2 + 0 \cdot 1 = 0 + 0 + 0 + 2 + 0 = 2
$$

#### Tại vị trí $ n = 1 $:
Kernel tiếp tục trùng với các phần tử khác trong tín hiệu:

$$
f[1] \cdot w_{\text{gập}}[0] + f[2] \cdot w_{\text{gập}}[1] + f[3] \cdot w_{\text{gập}}[2] + f[4] \cdot w_{\text{gập}}[3] + f[5] \cdot w_{\text{gập}}[4] = 0 \cdot 8 + 0 \cdot 2 + 1 \cdot 3 + 0 \cdot 2 + 0 \cdot 1 = 0 + 0 + 3 + 0 + 0 = 3
$$

#### Tại vị trí $ n = 2 $:

$$
f[2] \cdot w_{\text{gập}}[0] + f[3] \cdot w_{\text{gập}}[1] + f[4] \cdot w_{\text{gập}}[2] + f[5] \cdot w_{\text{gập}}[3] + f[6] \cdot w_{\text{gập}}[4] = 0 \cdot 8 + 1 \cdot 2 + 0 \cdot 3 + 0 \cdot 2 + 0 \cdot 1 = 0 + 2 + 0 + 0 + 0 = 2
$$

#### Tại vị trí $ n = 3 $:

$$
f[3] \cdot w_{\text{gập}}[0] + f[4] \cdot w_{\text{gập}}[1] + f[5] \cdot w_{\text{gập}}[2] + f[6] \cdot w_{\text{gập}}[3] + f[7] \cdot w_{\text{gập}}[4] = 1 \cdot 8 + 0 \cdot 2 + 0 \cdot 3 + 0 \cdot 2 + 0 \cdot 1 = 8 + 0 + 0 + 0 + 0 = 8
$$

#### Tại các vị trí còn lại:
Vì các phần tử tiếp theo của tín hiệu $ f $ đều là 0, kết quả của các phép chập sẽ là 0.

### Kết quả cuối cùng:

Kết quả phép chập 1 chiều là:

$$
[2, 3, 2, 8]
$$


# Phép chập - phiên bản 1
1. Trường hợp 1: `valid`
len(f) $\ge$ len(w)

kết quả của array phép chập đầu ra là

\begin{equation}
    \text{len} \left( \text{convoluted\_result} \right) = \text{len}\left(f\right) - \text{len}\left(w\right) + 1

\end{equation}

Trong trường hợp này, điểm bắt đầu chập là $f[0]$ với $w[0]$

2. Trường hợp 2: `same` (có padding nửa đầu nửa đuôi)

- phần này thì bù $\left\lfloor\frac{\text{len}({w})-1}{2}\right\rfloor$ mẫu $0$ vào `phía trước` của mảng f
- phần này thì bù $\left\lfloor\frac{\text{len}({w})-1}{2}\right\rfloor$ mẫu $0$ vào `phía sau` của mảng f

kết quả:
\begin{equation}
    \text{len}(\text{convoluted\_result}) = \text{len}(f)  
\end{equation}

3. Trường hợp 2: `full` (có padding nửa đầu nửa đuôi)

- phần này thì bù $\text{len}({w})-1$ mẫu $0$ vào `phía trước` của mảng f
- phần này thì bù $\text{len}({w})-1$ mẫu $0$ vào `phía trước` của mảng f
kết quả:
\begin{equation}
    \text{len}(\text{convoluted\_result}) = \text{len}(f) + \text{len}(w)-1
\end{equation}


In [1]:
import numpy as np
import tensorflow as tf
from scipy import signal

In [2]:
def convolve(f, w, mode='full'):
    if len(f) < len(w):
        raise ValueError("Độ dài của tín hiệu phải lớn hơn hoặc bằng độ dài của kernel.")

    # Gập kernel w (đảo ngược kernel)
    w_flipped = np.flip(w)

    # Tính kích thước của kết quả dựa trên mode
    if mode == 'full':
        output_size = len(f) + len(w) - 1
        padded_f = np.pad(f, (len(w)-1, len(w)-1), 'constant')  # Padding cho 'full'
    elif mode == 'same':
        output_size = len(f)
        pad_amount = (len(w) - 1) // 2
        padded_f = np.pad(f, (pad_amount, pad_amount), 'constant')  # Padding cho 'same'
    elif mode == 'valid':
        output_size = len(f) - len(w) + 1
        padded_f = f  # Không thêm padding cho 'valid'
    else:
        raise ValueError("Chế độ không hợp lệ. Vui lòng chọn 'full', 'same', hoặc 'valid'.")

    # Khởi tạo mảng kết quả với kích thước phù hợp
    convoluted_array = np.zeros(output_size)

    # Thực hiện phép tích chập thủ công
    for i in range(output_size):
        f_segment = padded_f[i:i + len(w_flipped)]
        convoluted_array[i] = (f_segment * w_flipped).sum()

    return convoluted_array
# Tín hiệu và kernel
f = np.array([0, 0, 0, 1, 0, 0, 0, 0])
w = np.array([1, 2, 3, 2, 8])

# Thực hiện tích chập với cả ba mode
result_full = convolve(f, w, mode='full')
result_same = convolve(f, w, mode='same')
result_valid = convolve(f, w, mode='valid')

print("Kết quả với mode 'full':", result_full)
print("Kết quả với mode 'same':", result_same)
print("Kết quả với mode 'valid':", result_valid)

Kết quả với mode 'full': [0. 0. 0. 1. 2. 3. 2. 8. 0. 0. 0. 0.]
Kết quả với mode 'same': [0. 1. 2. 3. 2. 8. 0. 0.]
Kết quả với mode 'valid': [2. 3. 2. 8.]


# Sử dụng phép chập trong thu viện numpy

In [3]:
f = np.array([0, 0, 0, 1, 0, 0, 0, 0])
w = np.array([1, 2, 3, 2, 8])
convolve(f,w,mode='valid')

array([2., 3., 2., 8.])

In [5]:
import numpy as np

# Tín hiệu và kernel
f = np.array([0, 0, 0, 1, 0, 0, 0, 0])
w = np.array([1, 2, 3, 2, 8])

# Thực hiện phép chập không padding với chế độ 'valid'
result = np.convolve(w,f, mode='valid')

print("Kết quả phép chập không padding:", result)

Kết quả phép chập không padding: [2 3 2 8]


# Sử dụng phép chập trong thử viện tensorflow

Ở thư viện tensorflow, có tham số stride: thể hiện việc trượt theo kiểu nhảy cóc

In [6]:
import tensorflow as tf
import numpy as np

# Tín hiệu đầu vào f và kernel w
f = np.array([0, 0, 0, 1, 0, 0, 0, 0], dtype=np.float32)
w = np.array([1, 2, 3, 2, 8], dtype=np.float32)

# Chuyển đổi tín hiệu f và kernel w thành dạng tensor với đúng shape
input_signal = f.reshape(1, -1, 1) 

kernel = w.reshape(-1, 1, 1)  # filter_width=5, in_channels=1, out_channels=1

# Thực hiện phép tích chập 1 chiều
output = tf.nn.conv1d(input_signal, kernel, stride=1, padding='SAME')

# Kết quả của tích chập
print("Kết quả của tích chập 1D:", output.numpy().flatten())

Kết quả của tích chập 1D: [0. 8. 2. 3. 2. 1. 0. 0.]


# Sử dụng phép chập trong thử viện scipy


In [7]:
import numpy as np
from scipy import signal

# Tín hiệu đầu vào và kernel
f = np.array([0, 0, 0, 1, 0, 0, 0, 0])
w = np.array([1, 2, 3, 2, 8])

# Thực hiện phép tích chập 1D sử dụng scipy.signal.convolve
result_full = signal.convolve(f, w, mode='full')
result_same = signal.convolve(f, w, mode='same')
result_valid = signal.convolve(f, w, mode='valid')

# In kết quả cho cả 3 mode
print("Kết quả với mode 'full':", result_full)
print("Kết quả với mode 'same':", result_same)
print("Kết quả với mode 'valid':", result_valid)

Kết quả với mode 'full': [0 0 0 1 2 3 2 8 0 0 0 0]
Kết quả với mode 'same': [0 1 2 3 2 8 0 0]
Kết quả với mode 'valid': [2 3 2 8]


# Bộ lọc FIR
## xử lý song song

![FIR](Pictures/pipeline_parallel_FIR.png)

* Đoạn code này mô tả hoạt động theo kiểu pipleline, nhân cộng song song toàn phần của một bộ lọc. Gồm các quá trình:
  - Dịch dần từng mẫu vào bộ lọc
  - thực hiện nhân với mẫu
  - thực hiện cộng các phép nhân

* Qua ví dụ này, ta sẽ thấy tại sao bản chất của bộ lọc là một phép chập:

        `array lọc + dịch tín hiệu = gập array lọc + dịch array đã gập`

In [9]:
def mac_filter(signal, kernel):

    filtered_value = []
    # Khởi tạo giá trị tích lũy
    accumulator = 0

    pad_amount = (len(kernel) - 1)
    # pad_amount = 0

    padded_signal = np.pad(signal, (0, pad_amount), 'constant')  # Padding cho 'full'
    print(padded_signal)
    

    # Khởi tạo một buffer có độ dài bằng kernel
    buffer = np.zeros(len(kernel))

    signal_len = len(signal)

    # Duyệt qua từng giá trị của tín hiệu đầu vào
    for i, val in enumerate(padded_signal):
        # Đẩy giá trị mới vào buffer (mô phỏng trượt của kernel)
        buffer = np.roll(buffer, 1)
        buffer[0] = val  # Thêm giá trị mới nhất vào buffer
        print(buffer)

        # Reset lại giá trị tích lũy
        accumulator = 0

        # Thực hiện tính MAC: nhân từng phần tử trong buffer với kernel và cộng dần vào accumulator
        for b, k in zip(buffer, kernel):
            accumulator += b * k

        filtered_value.append(accumulator)

    filtered_value = filtered_value[pad_amount//2: (pad_amount//2)+signal_len]

    return filtered_value

# Tín hiệu đầu vào và kernel
f = np.array([0, 0, 0, 1, 0, 0, 0, 0], dtype=np.float32)
w = np.array([1, 2, 3, 2, 8], dtype=np.float32)

# Khởi tạo bộ lọc MAC tuần tự
mac_generator = mac_filter(f, w)

print(mac_generator)

[0. 0. 0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0.]
[1. 0. 0. 0. 0.]
[0. 1. 0. 0. 0.]
[0. 0. 1. 0. 0.]
[0. 0. 0. 1. 0.]
[0. 0. 0. 0. 1.]
[0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0.]
[np.float64(0.0), np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(2.0), np.float64(8.0), np.float64(0.0), np.float64(0.0)]


## Sử dụng thư viện chuẩn của numpy để thực hiện bộ lọc FIR

In [10]:
from scipy import signal
import numpy as np

# Tạo tín hiệu và kernel
f = np.array([0, 0, 0, 1, 0, 0, 0, 0], dtype=np.float32)
b = np.array([1, 2, 3, 2, 8], dtype=np.float32)  # Hệ số bộ lọc

# Sử dụng lfilter để áp dụng bộ lọc lên tín hiệu
filtered_signal = signal.lfilter(b, [1.0], f)

print("Kết quả sử dụng scipy.signal.lfilter:", filtered_signal)

Kết quả sử dụng scipy.signal.lfilter: [0. 0. 0. 1. 2. 3. 2. 8.]


## Bộ lọc IIR đơn giản

\begin{equation}
x[n] = b[0]\cdot y[n] + b[1] \cdot y[n-1] + b[2] \cdot y[n-2] + \dots
\end{equation}

\begin{equation}
 \Leftrightarrow y[n] = x[n] - b[1] \cdot y[n-1] - b[2] \cdot y[n-2] - …
\end{equation}
![IIR](Pictures/pure_IIR.png)

In [11]:
def iir_filter(x,b):
    # Độ dài của hệ số b và a
    M = len(b)

    # Khởi tạo buffer cho các giá trị đầu vào (x) và đầu ra (y)
    x_buffer = np.zeros(1)  # Đệm cho giá trị đầu vào
    y_buffer = np.zeros(M)  # Đệm cho giá trị đầu ra

    # Kết quả đầu ra sau khi lọc
    y = np.zeros(len(x))

    # Áp dụng bộ lọc IIR thủ công
    for n in range(len(x)):
        # Cập nhật buffer đầu vào (x_buffer)
        x_buffer[0] = x[n]            # Thêm giá trị hiện tại vào đầu buffer

        # Reset giá trị đầu ra tạm thời
        y_temp = 0

        # Phần tích lũy từ đầu vào và các hệ số b
        y_temp += x_buffer[0]

        # Phần trừ từ các giá trị đầu ra trước đó và các hệ số a
        for j in range(1, M):
            y_temp -= b[j] * y_buffer[j]

        # Tính giá trị đầu ra y[n]
        y[n] = y_temp

        # Cập nhật buffer đầu ra (y_buffer)
        y_buffer[0] = y[n]            # Thêm giá trị mới nhất vào buffer

        y_buffer[1:] = y_buffer[:-1]  # Dịch chuyển buffer để chèn giá trị mới
        print(f'{n}-th signal: ')
        print(f'    + y_buffer = {y_buffer}')

    return y

# Tín hiệu đầu vào
x = np.array([1, 1, 2, 3, 0, 0, 0, 0], dtype=np.float32)

# Hệ số của bộ lọc IIR
b = np.array([ 1, 1, 1, 2.5], dtype=np.float32)

# Áp dụng bộ lọc IIR
filtered_signal = iir_filter(x, b)

# In kết quả
print("Tín hiệu sau khi lọc:", filtered_signal)

0-th signal: 
    + y_buffer = [1. 1. 0. 0.]
1-th signal: 
    + y_buffer = [0. 0. 1. 0.]
2-th signal: 
    + y_buffer = [1. 1. 0. 1.]
3-th signal: 
    + y_buffer = [-0.5 -0.5  1.   0. ]
4-th signal: 
    + y_buffer = [-0.5 -0.5 -0.5  1. ]
5-th signal: 
    + y_buffer = [-1.5 -1.5 -0.5 -0.5]
6-th signal: 
    + y_buffer = [ 3.25  3.25 -1.5  -0.5 ]
7-th signal: 
    + y_buffer = [-0.5  -0.5   3.25 -1.5 ]
Tín hiệu sau khi lọc: [ 1.    0.    1.   -0.5  -0.5  -1.5   3.25 -0.5 ]


## Bộ lọc IIR đơn giản bằng signal.lfilter

In [12]:
import numpy as np
from scipy import signal

# Tín hiệu đầu vào
x = np.array([1, 1, 2, 3, 0, 0, 0, 0], dtype=np.float32)

# Hệ số của bộ lọc IIR
a = np.array([1,1, 1, 2.5], dtype=np.float32)  # Hệ số của tử số
b = np.array([1.0], dtype=np.float32)  # Hệ số của mẫu số (để thực hiện bộ lọc FIR)

# Áp dụng bộ lọc IIR bằng lfilter
filtered_signal = signal.lfilter(b, a, x)

# In kết quả
print("Tín hiệu sau khi lọc bằng lfilter:", filtered_signal)

Tín hiệu sau khi lọc bằng lfilter: [ 1.    0.    1.   -0.5  -0.5  -1.5   3.25 -0.5 ]
